
# Group analysis with MNE-Python


The aim of this lecture is to show you how to do group analysis
with MNE-Python.

    Authors: Marijn van Vliet, Britta Westner, Alexandre Gramfort, Denis A. Engemann, Mainak Jas, Hicham Janati

    License: BSD (3-clause)
   

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

import numpy as np
import mne

mne.set_log_level("error")

# Change the following path to where you unzipped the extra data (`extra_meg_data-v2.zip`) on your disk.
extra_path = "../extra_data_mne"  # `./` means the folder of this notebook

evokeds_path = f"{extra_path}/group_analysis"

## Datasets

We have here the evoked data for all but 1 participant. That one dataset had some issues with the triggers that would have needed fixing to be read into MNE-Python, so here we just discard it (dataset #10).

In [ ]:
bad_subjects = [10]
datasets = ["sub-%02d" % i for i in range(1, 17) if i not in bad_subjects]
datasets

## Let's first look at the data from different subjects

We can look at the data from subjects one-by-one:

In [ ]:
subject = "sub-02"
evokeds = mne.read_evokeds(f"{evokeds_path}/{subject}_list-ave.fif", verbose=False)
evokeds

In [ ]:
# Fit a sphere to the headshape in order to make proper topo plots.
# This is needed for this particular dataset and may not be necessary for yours.
radius, center, _ = mne.bem.fit_sphere_to_headshape(evokeds[0].info, dig_kinds="eeg")
sphere = tuple(center) + (radius,)

evokeds[0].plot_joint(topomap_args=dict(sphere=sphere))

We can also loop over all subjects to get an overview and to see if there are any datasets
that look problematic.

In [ ]:
ch_type = "eeg"  # we pick the EEG channels here
conditions = ["famous", "unfamiliar", "scrambled"]

f, axes = plt.subplots(4, 4, figsize=(13, 9), sharex=True, sharey=True)

for ax, subject in zip(axes.ravel(), datasets):
    evokeds_dict = dict()
    evokeds = mne.read_evokeds(f"{evokeds_path}/{subject}_list-ave.fif")
    evokeds = [ev for ev in evokeds if ev.comment in conditions]
    for condition, ev in zip(conditions, evokeds):
        evokeds_dict[condition] = ev  #.crop(tmin=-0.3, tmax=0.6)
    mne.viz.plot_compare_evokeds(evokeds_dict, picks=ch_type, show=False,
                                 axes=ax, title=subject)

plt.tight_layout()

<div class='alert alert-success'>
    <b>EXERCISE</b>:
     <ul>
      <li>Compute the same type of GFP plots for MEG Magnetometers</li>
      <li>Do you see any problematic datasets?</li>
    </ul>
</div>

## Read all data and compute contrast

Next we will read in all datasets and collect them into one _nested_ list:

- sub01
  - famous
  - scrambled
- sub02
  - famous
  - scrambled
- ...

In [ ]:
evokeds_list = []

for subject in datasets:
    evokeds = mne.read_evokeds(f"{evokeds_path}/{subject}_list-ave.fif")
    evokeds = [ev for ev in evokeds if ev.comment in ["famous", "scrambled"]]
    evokeds_list.append(evokeds)

evokeds_list

From here, we can compute the contrast, e.g. between _famous_ and _scrambled_. We can also plot this contrast.

In [ ]:
contrast_list = []
f, axes = plt.subplots(4, 4, figsize=(13, 9), sharex=True, sharey=True)

for ax, subject, evokeds in zip(axes.ravel(), datasets, evokeds_list):
    contrast = mne.combine_evoked(evokeds, weights=[0.5, -0.5])
    contrast.comment = "contrast"
    contrast_list.append(contrast)
    mne.viz.plot_compare_evokeds(contrast, picks=ch_type, show=False,
                                 axes=ax, title=subject, legend=False)
plt.tight_layout()

In [ ]:
contrast_list

<div class='alert alert-success'>
    <b>EXERCISE</b>:
     <ul>
      <li>Compute the contrast between the "famous" versus "unfamiliar" conditions.</li>
      <li>Optional: change the code back to the original "famous" versus "scambled" condition if you want to follow along exactly with what is happening on stage.</li>
    </ul>
</div>

### Look at grand averages

Grand averages are obtained by averaging the sensor space data.

In [ ]:
mne.grand_average?

In [ ]:
contrast_gave = mne.grand_average(contrast_list)
contrast_gave.plot_joint(topomap_args=dict(sphere=sphere));

Grand averages for each condition

In [ ]:
condition1_gave = mne.grand_average([evokeds[0] for evokeds in evokeds_list])
condition2_gave = mne.grand_average([evokeds[1] for evokeds in evokeds_list])
condition1_gave.comment = "famous"
condition2_gave.comment = "scrambled"
mne.viz.plot_compare_evokeds([condition1_gave, condition2_gave])

<div class='alert alert-success'>
    <b>EXERCISE</b>:
     <ul>
      <li>Compute the grand averages of the "famous" versus "unfamiliar" conditions, and plot them.</li>
      <li>Optional: change the code back to the original "famous" versus "scambled" condition if you want to follow along exactly with what is happening on stage.</li>
    </ul>
</div>

## Let's do some statistics - t-maps

MNE-Python's statistics API is currently quite low level, operating on numpy arrays. To do statistical, we will be getting the data from an MNE-Python object to a numpy array, and putting a numpy array in an MNE-Python object.

### Sensor level t-map

We start by getting the data from the `Evoked` objects containing for each subject the contrasts between conditions, and concatenating it as one big numpy array:

In [ ]:
import numpy as np
X = np.array([contrast.data for contrast in contrast_list])
X.shape

Then, we can do a t-test against 0:

In [ ]:
from mne.stats import ttest_1samp_no_p
tvalues = ttest_1samp_no_p(X)
tvalues.shape

To visualize the t-values, we need to pack them into an MNE-Python `Evoked` object:

In [ ]:
mne.EvokedArray?

In [ ]:
tmap = mne.EvokedArray(tvalues, info=contrast_gave.info, tmin=contrast_gave.tmin, comment="tmap")
tmap.plot_joint(topomap_args=dict(sphere=sphere), ts_args=dict(scalings=1, units="t-value"))

## Cluster-permutation test on one channel

We'll start with a single channel, EEG065. 

We will run a cluster-permutation test of our contrast against 0.

Let's start with the setup.

Let's get our data into a numpy array so we can feed it to the stats function.

In [ ]:
channel = 'EEG065'
ch_idx = contrast_gave.ch_names.index(channel)
X_single_chan = X[:, ch_idx, :]

Now, let's set some parameters for our test:

In [ ]:
n_jobs = 2  # number of parallel jobs, you might want to set this higher (or to 1 to disable multithreading)
n_permutations = 5000  # number of permutations to run 

# specify adjacency of points in the data: 
# here, no special adjacency is needed because it's only one channel and MNE will now
# just assume a regular grid, which is fine for our time dimension
adjacency = None  

tail = 0.  # we will run a two-sided test

Okay, with that out of the way, let's set our cluster thresholds based on the tmap we created before:

In [ ]:
threshold = 2.85

Now we have everything to run our statistics!

In [ ]:
cluster_stats = mne.stats.permutation_cluster_1samp_test(
    X_single_chan, 
    threshold=threshold,
    n_jobs=n_jobs, 
    verbose=True, 
    tail=tail,
    adjacency=adjacency,
    n_permutations=n_permutations, 
    seed=42,)

T_obs, clusters, cluster_p_values, _ = cluster_stats

#### Visualize results from one-channel test

In [ ]:
# set up the figure
fig, axes = plt.subplots(2, sharex=True)

# on first axis: plot the data (contrast averaged across datasets)
ax = axes[0]
ax.plot(contrast.times, X_single_chan.mean(axis=0), label="ERP Contrast")
ax.set(title=f"Channel: {channel}", ylabel="μV")
ax.legend()

# on second axis: plot the clusters.

ax = axes[1]
# enumerate across the clusters
for ii, cluster_ii in enumerate(clusters):
    c = cluster_ii[0]

    # check if the matching p-value is smaller than some critical value:
    # if so, color the stretch of time red.
    critical_value = 0.05
    if cluster_p_values[ii] < critical_value:
        fill_color = "orange"
    else:
        fill_color = "white"
        
    h1 = ax.axvspan(contrast.times[c[0]], contrast.times[c[-1] - 1],
                    fc=fill_color, ec="black", alpha=0.3)

# plot the t-values 
hf = ax.plot(contrast.times, T_obs, color="green")

# set legend, axes, and show
ax.legend((h1,), (f"p < {critical_value}",), loc="upper right", ncol=1)
ax.set(xlabel="Time (s)", ylabel="t-values",
       ylim=[-10., 10.], xlim=contrast.times[[0, -1]])
fig.tight_layout(pad=0.5)
plt.show()

<div class='alert alert-success'>
    <b>EXERCISE</b>:
     <ul>
      <li>Perform the cluster-based permutation test between the "famous" versus "unfamiliar" conditions. Do this on the same channel as we did above. You fill find that in this case, the threshold of 2.85 may be too high. Try with a lower threshold.</li>
      <li>Optional: Change the code back to the original "famous" versus "scambled" condition if you want to follow along exactly with what is happening on stage.</li>
    </ul>
</div>

## Cluster-permutation test across time and sensors

Now let's not choose a sensor, but run the clustering approach across all EEG sensors!

In [ ]:
# here we can use a convenience function for spatio-temporal data:
from mne.stats import spatio_temporal_cluster_1samp_test

In [ ]:
# let's use EEG here:
ch_type = "eeg"

# We again need to make our data into an array 
X = np.array([contrast.copy().pick(ch_type).data
              for contrast in contrast_list])
X.shape

The clustering function tells us:
```
X : array, shape (n_observations, p[, q], n_vertices)
    The data to be clustered. The first dimension should correspond to the
    difference between paired samples (observations) in two conditions.
    The second, and optionally third, dimensions correspond to the
    time or time-frequency data. And, the last dimension should be spatial.
```

In [ ]:
# Make sure spatial dimension is last:
X = np.transpose(X, (0, 2, 1)) 
X.shape

Now our parameters again. This time we need to pay close attentiont to the adjacency!

In [ ]:
# tail and cluster thresholds
tail = 0.  # for two sided test

# set cluster threshold (see above for more detail)
threshold = 2.85

# Make a triangulation between MEG sensor locations to
# use as adjacency informatiom for cluster level stats:
adjacency = mne.channels.find_ch_adjacency(contrast.info, ch_type)[0]

In [ ]:
cluster_stats = spatio_temporal_cluster_1samp_test(
    X, 
    threshold=threshold, 
    n_jobs=2, 
    verbose=True, 
    tail=tail,
    adjacency=adjacency, 
    out_type='indices',
    check_disjoint=True, 
    seed=42)

In [ ]:
# Let's make the output easier to use for plotting:
T_obs, clusters, p_values, _ = cluster_stats
good_cluster_inds = np.where(p_values < 0.05)[0]

print("Good clusters: %i" % len(good_cluster_inds))

In [ ]:
clusters[good_cluster_inds[0]]

#### Visualize the spatio-temporal clusters

In [ ]:
from mpl_toolkits.axes_grid1 import make_axes_locatable
from mne.viz import plot_topomap

# find the relevant sensors
pos = mne.find_layout(contrast.info, ch_type=ch_type).pos

T_obs_max = 5.
T_obs_min = -T_obs_max

# loop over significant clusters
for i_clu, clu_idx in enumerate(good_cluster_inds):

    # unpack cluster information, get unique indices per cluster
    time_inds, space_inds = np.squeeze(clusters[clu_idx])
    ch_inds = np.unique(space_inds)
    time_inds = np.unique(time_inds)

    # get topography for stats and average across time
    T_obs_map = T_obs[time_inds, ...].mean(axis=0)

    # get signal at significant sensors and average across subjects and sensors
    signal = X[..., ch_inds].mean(axis=(0, 2))
    sig_times = contrast.times[time_inds]

    # create a spatial mask
    mask = np.zeros((T_obs_map.shape[0], 1), dtype=bool)
    mask[ch_inds, :] = True

    # initialize figure
    fig, ax_topo = plt.subplots(1, 1, figsize=(7, 2.))

    # how to mark sensors in the cluster
    mask_params = dict(marker='.', markerfacecolor='k', markersize=2)

    # plot average test statistic and mark significant sensors
    sel = mne.pick_types(contrast.info, meg=False, eeg=True)
    info = mne.pick_info(contrast.info, sel)
    image, _ = plot_topomap(T_obs_map, info, ch_type=ch_type, mask=mask, 
                            axes=ax_topo, sensors=False, sphere=sphere,
                            mask_params=mask_params,
                            vlim=(T_obs_min, T_obs_max),
                            show=False)

    # advanced matplotlib for showing image with figure and colorbar
    # in one plot
    divider = make_axes_locatable(ax_topo)

    # add axes for colorbar, add colorbar 
    ax_colorbar = divider.append_axes("right", size="5%", pad=0.05)
    plt.colorbar(image, cax=ax_colorbar, format="%0.1f")

    # label the topoplot
    ax_topo.set_xlabel(
        f"Averaged t-map\n({sig_times[0]:0.2f} - {sig_times[1]:0.2f} ms)"
    )

    # add a new axis for time courses and plot time courses
    ax_signal = divider.append_axes("right", size="300%", pad=1.2)
    ax_signal.plot(contrast.times, signal)

    # mark stimulus onset
    ax_signal.axvline(0, color="k", linestyle=":")

    # adjust and label axes
    ax_signal.set_xlim([contrast.times[0], contrast.times[-1]])
    ax_signal.set_xlabel("Time (s)")
    ax_signal.set_ylabel("ERP Contrast (μV)")

    # plot the significant time range in time course
    ymin, ymax = ax_signal.get_ylim()
    ax_signal.fill_betweenx((ymin, ymax), sig_times[0], sig_times[-1],
                            color="orange", alpha=0.3)
    title = f"Cluster #{i_clu + 1} (p < {p_values[clu_idx]:0.3f})"
    ax_signal.set(ylim=[ymin, ymax], title=title)

    # clean up the figure a little
    fig.tight_layout(pad=0.5, w_pad=0)
    fig.subplots_adjust(bottom=.05)

plt.show()

<div class='alert alert-success'>
    <b>EXERCISE</b>:
     <ul>
      <li>Perform the cluster-based permutation test between the "famous" versus "unfamiliar" conditions. You fill find that in this case, the threshold of 2.85 may be too high. Try with a lower threshold.</li>
    </ul>
</div>

#### Further reading and credit:

The code in this notebook is heavily inspired by https://github.com/mne-tools/mne-biomag-group-demo/tree/master/scripts/results/statistics

More about sensor level statistics can be found here: https://mne.tools/stable/auto_tutorials/stats-sensor-space/index.html

If you want to do stats in source space, check this: https://mne.tools/stable/auto_tutorials/stats-source-space/index.html